#### Zadanie: Denoising autoencoder

W klasycznym autoenkoderze sieć uczy się: Wejście -> Enkoder -> Wąskie Gardło -> Dekoder -> Wyjście = Wejście.

W odszumiającym autoenkoderze wprowadzamy drobną zmianę:
1. Bierzemy oryginalny obrazek ($X$).
2. Dodajemy do niego sztuczny szum, tworząc zniszczony obrazek ($X_{noisy}$).
3. Podajemy zniszczony obrazek do sieci: Wyjście = Model(X_noisy).
4. Kluczowy moment: Błąd (Loss) obliczamy porównując Wyjście z oryginalnym, czystym obrazkiem ($X$), a nie z obrazkiem zaszumionym!

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [7]:
# Konfiguracja urządzenia
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Urządzenie: {DEVICE}")

Urządzenie: cpu


In [9]:
# 1. Przygotowanie danych
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # Spłaszczanie obrazka 28x28 do wektora 784
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

In [11]:
# =====================================================================
# ZADANIE 1: Funkcja dodająca szum
# =====================================================================
def add_noise(images, noise_factor=0.3):
    """
    Funkcja przyjmuje czyste obrazy i dodaje do nich losowy szum.
    Szum powinien być losowany z rozkładu normalnego.
    
    Wskazówka 1: Użyj torch.randn_like(images), aby wygenerować szum.
    Wskazówka 2: Pomnóż szum przez noise_factor, aby kontrolować jego siłę.
    Wskazówka 3: Po dodaniu szumu, wartości pikseli mogą przekroczyć zakres [0, 1]. 
                 Użyj torch.clamp(tensor, 0., 1.), aby je przyciąć.
    """
    
    ### TODO: Zaimplementuj dodawanie szumu tutaj ###
    
    noisy_images = None # Zastąp to swoim kodem
    
    return noisy_images

In [15]:
# =====================================================================
# ZADANIE 2: Architektura modelu
# =====================================================================
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()
        
        ### TODO: Zbuduj Enkoder (np. 784 -> 256 -> 128 -> 64) ###
        self.encoder = nn.Sequential(
            # Twój kod tutaj
        )
        
        ### TODO: Zbuduj Dekoder (np. 64 -> 128 -> 256 -> 784) ###
        # Pamiętaj o użyciu odpowiedniej funkcji aktywacji na samym końcu!
        self.decoder = nn.Sequential(
            # Twój kod tutaj
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Inicjalizacja
model = DenoisingAutoencoder().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

ValueError: optimizer got an empty parameter list

In [17]:
# =====================================================================
# ZADANIE 3: Pętla Treningowa
# =====================================================================
num_epochs = 10
print("Rozpoczynamy trening...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for data, _ in train_loader:
        clean_images = data.to(DEVICE)
        
        ### TODO: 1. Stwórz zaszumione obrazy używając swojej funkcji add_noise ###
        noisy_images = None # Twój kod
        
        ### TODO: 2. Wyzeruj gradienty optymalizatora ###
        
        ### TODO: 3. Przepuść ZASZUMIONE obrazy przez model ###
        outputs = None # Twój kod
        
        ### TODO: 4. Oblicz błąd (Loss). 
        # UWAGA: Porównaj wyjście z sieci z CZYSTYMI obrazami! ###
        loss = None # Twój kod
        
        ### TODO: 5. Wykonaj propagację wsteczną (backward) i krok optymalizatora ###
        
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoka [{epoch+1}/{num_epochs}], Średnia Strata: {avg_loss:.4f}')

Rozpoczynamy trening...


AttributeError: 'NoneType' object has no attribute 'item'

In [ ]:
# =====================================================================
# ZADANIE 4: Ewaluacja i Wizualizacja
# =====================================================================
model.eval()

# Pobieramy jedną paczkę danych testowych
test_data, _ = next(iter(test_loader))
clean_test_images = test_data.to(DEVICE)

# Tworzymy zaszumione wersje do testów
noisy_test_images = add_noise(clean_test_images, noise_factor=0.3)

with torch.no_grad():
    ### TODO: Przepuść noisy_test_images przez wytrenowany model ###
    reconstructed_images = None # Twój kod

# --- WIZUALIZACJA (Nie zmieniaj tego kodu) ---
# Jeśli wszystko zrobiłeś poprawnie, ten kod wygeneruje 3 rzędy obrazków:
# 1. Oryginalne | 2. Zaszumione | 3. Zrekonstruowane (Odszumione)

clean_test_images = clean_test_images.cpu().view(-1, 28, 28).numpy()
noisy_test_images = noisy_test_images.cpu().view(-1, 28, 28).numpy()
reconstructed_images = reconstructed_images.cpu().view(-1, 28, 28).numpy()

fig, axes = plt.subplots(nrows=3, ncols=10, figsize=(15, 5))
for i in range(10):
    # Rząd 1: Oryginał
    axes[0, i].imshow(clean_test_images[i], cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_title("Oryginał")
        
    # Rząd 2: Zaszumione
    axes[1, i].imshow(noisy_test_images[i], cmap='gray')
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_title("Zaszumione")
        
    # Rząd 3: Rekonstrukcja
    axes[2, i].imshow(reconstructed_images[i], cmap='gray')
    axes[2, i].axis('off')
    if i == 0: axes[2, i].set_title("Odszumione")

plt.tight_layout()
plt.show()